<a href="https://colab.research.google.com/github/damianwgriggs/Quantum-Crypto-Predictor/blob/main/Crypto_Predictor_Backtest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
# ------------------------------------------------------------------------------
# 🚀 FINAL "REALISTIC GMX" BACKTESTER (COMPOUNDING + 5x LEVERAGE)
# ------------------------------------------------------------------------------
import os
import sys

# --- STEP 1: CHECK ENVIRONMENT ---
print("⏳ CHECKING QUANTUM SILO...")
if not os.path.exists('/usr/local/envs/quantum_env'):
    os.system('wget -qO mini.sh https://repo.anaconda.com/miniconda/Miniconda3-py39_23.3.1-0-Linux-x86_64.sh')
    os.system('chmod +x mini.sh')
    os.system('bash ./mini.sh -b -f -p /usr/local > /dev/null')
    os.system('/usr/local/bin/conda create -y -n quantum_env python=3.9 > /dev/null')
    os.system('/usr/local/envs/quantum_env/bin/pip install -q tensorflow==2.15.0 tensorflow-quantum==0.7.3 cirq==1.3.0 sympy==1.12 ccxt ta pandas requests tqdm pytz yfinance > /dev/null')
os.system('/usr/local/envs/quantum_env/bin/pip install -q yfinance > /dev/null')
print("✅ ENVIRONMENT READY.")

# --- STEP 2: WRITE THE COMPOUNDING SCRIPT ---
script_content = """
import os
import sys
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_quantum as tfq
import cirq
import sympy
import ta
import pytz
import yfinance as yf
import warnings

warnings.filterwarnings("ignore")
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# --- CONFIGURATION ---
SYMBOL_YF = 'ETH-USD'
TIMEFRAME = '1h'
DATA_PERIOD = '2y'
TEST_RATIO = 0.20
CONFIDENCE_THRESHOLD = 0.55

# GMX STRATEGY
TARGET_ROI = 0.012   # 1.2% Profit
STOP_LOSS = 0.008    # 0.8% Loss
GMX_FEE = 0.001      # 0.1% Open + 0.1% Close = 0.2% Total

# LEVERAGE SETTINGS
LEVERAGE = 5.0             # 5x Leverage
COLLATERAL_PCT = 0.20      # Use 20% of Account as Collateral
# Effective Position Size = Balance * 0.20 * 5 = 100% of Balance

# 1. FETCH DATA
print(f"\\n📡 FETCHING {DATA_PERIOD} OF {SYMBOL_YF} DATA...")
try:
    df = yf.download(tickers=SYMBOL_YF, period=DATA_PERIOD, interval=TIMEFRAME, progress=False)
    if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)
    df.rename(columns={'Open': 'open', 'High': 'high', 'Low': 'low', 'Close': 'close', 'Volume': 'volume'}, inplace=True)
    df.index = pd.to_datetime(df.index, utc=True)
    df = df.tz_convert('America/Los_Angeles')

    # --- INDICATORS ---
    df['RSI'] = ta.momentum.RSIIndicator(df['close'], window=14).rsi()
    macd = ta.trend.MACD(df['close'])
    df['MACD_Diff'] = macd.macd_diff()
    df['Vol_SMA'] = df['volume'].rolling(window=20).mean()
    df['Vol_Ratio'] = df['volume'] / df['Vol_SMA']

    # --- TARGETS ---
    look_forward = 10
    targets = []

    for i in range(len(df) - look_forward):
        entry_price = df.iloc[i]['close']
        future_window = df.iloc[i+1 : i+1+look_forward]
        max_high = future_window['high'].max()
        min_low = future_window['low'].min()

        long_win = max_high >= (entry_price * (1 + TARGET_ROI))
        long_loss = min_low <= (entry_price * (1 - STOP_LOSS))
        short_win = min_low <= (entry_price * (1 - TARGET_ROI))
        short_loss = max_high >= (entry_price * (1 + STOP_LOSS))

        if long_win and not long_loss: targets.append(1)
        elif short_win and not short_loss: targets.append(0)
        else: targets.append(np.nan)

    df = df.iloc[:len(targets)]
    df['Target'] = targets

    # --- FILTER (Active Hours) ---
    active_hours = [6, 7, 8, 9, 10, 16, 17, 18, 19, 20]
    active_df = df[(df.index.dayofweek <= 4) & (df.index.hour.isin(active_hours))].dropna()

    # Normalize
    active_df['n_RSI'] = active_df['RSI'] / 100.0
    active_df['n_MACD'] = (active_df['MACD_Diff'] - active_df['MACD_Diff'].min()) / (active_df['MACD_Diff'].max() - active_df['MACD_Diff'].min())
    active_df['n_Vol'] = (active_df['Vol_Ratio'] - active_df['Vol_Ratio'].min()) / (active_df['Vol_Ratio'].max() - active_df['Vol_Ratio'].min())
    active_df['n_Mom'] = np.where(active_df['close'] > active_df['open'], 0.8, 0.2)

    print(f"✅ Data Processed: {len(active_df)} Trading Windows.")

except Exception as e:
    print(f"❌ Data Error: {e}")
    sys.exit(1)

# 2. SPLIT & ENCODE
split_idx = int(len(active_df) * (1 - TEST_RATIO))
train_df = active_df.iloc[:split_idx]
test_df = active_df.iloc[split_idx:]

qubits = [cirq.GridQubit(0, i) for i in range(4)]

def process_to_circuits(dataframe):
    circuits = []
    labels = []
    for _, row in dataframe.iterrows():
        c = cirq.Circuit()
        c.append(cirq.ry(row['n_RSI'] * np.pi)(qubits[0]))
        c.append(cirq.ry(row['n_MACD'] * np.pi)(qubits[1]))
        c.append(cirq.ry(row['n_Vol'] * np.pi)(qubits[2]))
        c.append(cirq.ry(row['n_Mom'] * np.pi)(qubits[3]))
        circuits.append(c)
        labels.append(row['Target'])
    return tfq.convert_to_tensor(circuits), np.array(labels)

X_train, y_train = process_to_circuits(train_df)
X_test, y_test = process_to_circuits(test_df)

# 3. TRAIN MODEL
params = sympy.symbols('theta0:14')
q_model_circuit = cirq.Circuit()
q_model_circuit.append(cirq.rx(params[0])(qubits[0]))
q_model_circuit.append(cirq.ry(params[1])(qubits[1]))
q_model_circuit.append(cirq.rx(params[2])(qubits[2]))
q_model_circuit.append(cirq.ry(params[3])(qubits[3]))
q_model_circuit.append(cirq.CZ(qubits[0], qubits[1]))
q_model_circuit.append(cirq.CZ(qubits[2], qubits[3]))
q_model_circuit.append(cirq.ry(params[4])(qubits[0]))
q_model_circuit.append(cirq.rx(params[5])(qubits[1]))
q_model_circuit.append(cirq.ry(params[6])(qubits[2]))
q_model_circuit.append(cirq.rx(params[7])(qubits[3]))
q_model_circuit.append(cirq.CZ(qubits[1], qubits[2]))
q_model_circuit.append(cirq.rx(params[8])(qubits[0]))
q_model_circuit.append(cirq.ry(params[9])(qubits[1]))
q_model_circuit.append(cirq.rx(params[10])(qubits[2]))
q_model_circuit.append(cirq.rx(params[11])(qubits[3]))
q_model_circuit.append(cirq.ry(params[12])(qubits[0]))
q_model_circuit.append(cirq.rx(params[13])(qubits[0]))

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(), dtype=tf.string),
    tfq.layers.PQC(q_model_circuit, operators=cirq.Z(qubits[0])),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.005), loss='binary_crossentropy', metrics=['accuracy'])
print("\\n🧠 TRAINING (REALISTIC SIZING)...")
model.fit(X_train, y_train, epochs=40, batch_size=32, verbose=0)

# 4. RESULTS
print(f"\\n💰 REALISTIC GMX BACKTEST (5x Leverage, 20% Collateral)")
print("-" * 100)
print(f"{'Date':<15} | {'Price':<8} | {'Position Size':<14} | {'Res':<4} | {'PnL ($)':<10} | {'Balance'}")
print("-" * 100)

predictions = model.predict(X_test, verbose=0)
balance = 10000
wins, losses, skipped = 0, 0, 0

for i in range(len(test_df)):
    row = test_df.iloc[i]
    prob = predictions[i][0]

    pred_dir = 1 if prob > 0.5 else 0
    actual = row['Target']
    confidence = prob if prob > 0.5 else (1 - prob)

    if confidence < CONFIDENCE_THRESHOLD:
        skipped += 1
        continue

    # --- DYNAMIC POSITION SIZING ---
    # We use 20% of CURRENT Balance as Collateral
    collateral = balance * COLLATERAL_PCT
    # We apply 5x Leverage
    position_size = collateral * LEVERAGE

    is_win = False
    if pred_dir == 1 and actual == 1: is_win = True
    elif pred_dir == 0 and actual == 0: is_win = True

    # FEES: Apply to the TOTAL Position Size (Entry + Exit = 0.2%)
    fee_cost = position_size * (GMX_FEE * 2)

    pnl = 0
    res_str = ""

    if is_win:
        # Profit on Total Position Size
        gross_profit = position_size * TARGET_ROI
        pnl = gross_profit - fee_cost
        balance += pnl
        wins += 1
        res_str = "WIN"
    else:
        # Loss on Total Position Size
        gross_loss = position_size * STOP_LOSS
        pnl = -(gross_loss + fee_cost)
        balance += pnl
        losses += 1
        res_str = "LOSS"

    # Print Trades
    if i > len(test_df) - 40:
        ts = row.name.strftime('%m-%d %H:%M')
        print(f"{ts:<15} | {row['close']:<8.0f} | ${position_size:<13.0f} | {res_str:<4} | {pnl:>10.2f} | ${balance:,.2f}")

print("-" * 100)
total = wins + losses
acc = (wins / total * 100) if total > 0 else 0
roi_pct = ((balance - 10000) / 10000) * 100

print(f"🏁 FINAL RESULTS:")
print(f"   Start Balance: $10,000")
print(f"   Final Balance: ${balance:,.2f}")
print(f"   Net Profit:    ${balance - 10000:,.2f} (+{roi_pct:.1f}%)")
print(f"   Win Rate:      {acc:.1f}%")
print(f"   Trades Taken:  {total}")
"""

with open("run_gmx_realistic.py", "w") as f:
    f.write(script_content)

# --- STEP 3: EXECUTE ---
print("🚀 LAUNCHING REALISTIC SIMULATION...")
!/usr/local/envs/quantum_env/bin/python run_gmx_realistic.py

⏳ CHECKING QUANTUM SILO...
✅ ENVIRONMENT READY.
🚀 LAUNCHING REALISTIC SIMULATION...
2025-12-02 16:36:55.812277: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-12-02 16:36:55.812343: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-12-02 16:36:55.813827: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-12-02 16:36:55.822618: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with t